In [1]:
import pandas as pd
import yfinance as yf
import os
import re
import numpy as np

Create Directory if needed

In [2]:
os.makedirs(os.path.dirname("./saveData"), exist_ok=True)

Download Twitter Data

In [3]:
sentiment_path = "./savedData/datasets/gauravduttakiit/bitcoin-tweets-16m-tweets-with-sentiment-tagged/versions/1/mbsa.csv"

if os.path.exists(sentiment_path):
    df_sentiment = pd.read_csv(sentiment_path)
    print("Sentiment data loaded")
else:
    os.environ["KAGGLEHUB_CACHE"] = "./savedData"
    import kagglehub

    # Download the Bitcoin tweet dataset
    path = kagglehub.dataset_download("gauravduttakiit/bitcoin-tweets-16m-tweets-with-sentiment-tagged")
    print("Path to dataset files:", path)
    df_sentiment = pd.read_csv(sentiment_path)
    print("Sentiment data downloaded, saved and loaded")

Sentiment data loaded


Data Pipeline

In [4]:
# Delete Duplicates
df_sentiment = df_sentiment.drop_duplicates()

# Clean Dates
df_sentiment = df_sentiment[pd.to_datetime(df_sentiment['Date'], errors='coerce').notnull()]
df_sentiment['Date'] = pd.to_datetime(df_sentiment['Date'], errors='coerce')

# Clean and map Sentiment
sentiment_map = {'Positive': 1, 'Negative': -1}
df_sentiment = df_sentiment.dropna(subset=['Sentiment'])
df_sentiment = df_sentiment[df_sentiment['Sentiment'].isin(['Positive', 'Negative'])].copy()
df_sentiment['sentiment_score'] = df_sentiment['Sentiment'].map(sentiment_map)

# Clean Text
df_sentiment = df_sentiment.dropna(subset=['text'])

In [5]:
df_sentiment.head()

,Date,text,Sentiment,sentiment_score
0,2019-05-27,È appena uscito un nuovo video! LES CRYPTOMONN...,Positive,1
1,2019-05-27,Cardano: Digitize Currencies; EOS https://t.co...,Positive,1
2,2019-05-27,Another Test tweet that wasn't caught in the s...,Positive,1
3,2019-05-27,Current Crypto Prices! \n\nBTC: $8721.99 USD\n...,Positive,1
4,2019-05-27,Spiv (Nosar Baz): BITCOIN Is An Asset &amp; NO...,Positive,1


Optional: Remove emojis and url´s from tweets

In [6]:
# df_sentiment.loc[:, 'text'] = df_sentiment['text'].apply(lambda x: re.sub(r"http\S+|www\S+|https\S+", '', x))
# df_sentiment.loc[:, 'text'] = df_sentiment['text'].apply(lambda x: re.sub(r'[^\w\s]', '', x))

In [7]:
start_date = df_sentiment['Date'].min()
end_date = df_sentiment['Date'].max()
print("Minimales Datum: ", start_date, "\nMaximales Datum: ", end_date)

Minimales Datum:  2014-09-18 00:00:00 
Maximales Datum:  2019-11-23 00:00:00


Download Bitcoin Data

In [8]:
bitcoin_path = "./savedData/bitcoinPrice.csv"

if(os.path.exists(bitcoin_path)):
    df_bitcoin = pd.read_csv(bitcoin_path, header=0)
    df_bitcoin = df_bitcoin.dropna(subset=["Date"])
    df_bitcoin['Date'] = pd.to_datetime(df_bitcoin['Date'])
    df_bitcoin.columns = ['Date', 'Close']
    print("Bitcoin data loaded")
else:
    ticker = "BTC-USD"
    df_bitcoin = yf.download(ticker, start=start_date, end=end_date, auto_adjust=False)
    df_bitcoin.reset_index(inplace=True)
    df_bitcoin = df_bitcoin[['Date', 'Close']]
    df_bitcoin['Date'] = pd.to_datetime(df_bitcoin['Date'])
    df_bitcoin.columns = ['Date', 'Close']
    df_bitcoin.to_csv(bitcoin_path, index=False)
    print("Bitcoin data downloaded, loaded and saved")

Bitcoin data loaded


In [9]:

df_bitcoin.head()

,Date,Close
1,2014-09-18,424.44000244140625
2,2014-09-19,394.7959899902344
3,2014-09-20,408.90399169921875
4,2014-09-21,398.8210144042969
5,2014-09-22,402.1520080566406


In [10]:
# Calculate sentiment per day
df_daily_sentiment = df_sentiment.groupby('Date')['sentiment_score'].mean().reset_index()
df_daily_sentiment.rename(columns={'sentiment_score': 'avg_sentiment'}, inplace=True)
df_daily_sentiment['Date'] = pd.to_datetime(df_daily_sentiment['Date'])

In [11]:
df_daily_sentiment.head()

,Date,avg_sentiment
0,2014-09-18,-1.0
1,2014-09-19,-1.0
2,2014-09-20,1.0
3,2014-09-21,-1.0
4,2014-09-22,1.0


In [12]:
# Merge daily Bitcoin price and daily Sentiment, sort it and rename
merged = pd.merge(df_daily_sentiment, df_bitcoin, on='Date', how='inner')
merged.sort_values('Date', inplace=True)
merged['next_day_price'] = merged['Close'].shift(-1)
merged['target_trend'] = np.where(merged['next_day_price'] > merged['Close'], 1, -1)

In [13]:
merged.head()

,Date,avg_sentiment,Close,next_day_price,target_trend
0,2014-09-18,-1.0,424.44000244140625,394.7959899902344,-1
1,2014-09-19,-1.0,394.7959899902344,408.90399169921875,1
2,2014-09-20,1.0,408.90399169921875,398.8210144042969,-1
3,2014-09-21,-1.0,398.8210144042969,402.1520080566406,1
4,2014-09-22,1.0,402.1520080566406,435.7909851074219,1


In [14]:
# Easy Clssification for now
merged['prediction'] = np.where(merged['avg_sentiment'] > 0, 1, -1)
accuracy = (merged['prediction'] == merged['target_trend']).mean()

print('Vergleich von Sentiment und Preistrend für alle Tage')
print(f'Übereinstimmungsquote: {accuracy:.3f}')
print(merged[['Date', 'avg_sentiment', 'Close', 'next_day_price', 'target_trend', 'prediction']].head(10))

Vergleich von Sentiment und Preistrend für alle Tage
Übereinstimmungsquote: 0.488
        Date  avg_sentiment               Close      next_day_price  \
0 2014-09-18           -1.0  424.44000244140625   394.7959899902344   
1 2014-09-19           -1.0   394.7959899902344  408.90399169921875   
2 2014-09-20            1.0  408.90399169921875   398.8210144042969   
3 2014-09-21           -1.0   398.8210144042969   402.1520080566406   
4 2014-09-22            1.0   402.1520080566406   435.7909851074219   
5 2014-09-23            1.0   435.7909851074219   423.2049865722656   
6 2014-09-24           -1.0   423.2049865722656   411.5740051269531   
7 2014-09-25           -1.0   411.5740051269531  404.42498779296875   
8 2014-09-26           -1.0  404.42498779296875   399.5199890136719   
9 2014-09-27           -1.0   399.5199890136719   377.1809997558594   

   target_trend  prediction  
0            -1          -1  
1             1          -1  
2            -1           1  
3             1 